# Soccer Player Value Visualization Menu

This notebook is a **menu, not a fixed pipeline**. Every section below makes ONE type of chart about the same dataset (player stats + transfer value, La Liga / Premier League / Serie A / Bundesliga / Ligue 1, 2017-2020).

**How to use this notebook:**
1. Run the **Setup** section once (loads the data, do this every time).
2. Skim the menu below the setup section — each one has a one-line description and a small preview image of the *shape* of the chart (not the real data).
3. Jump to whichever section looks interesting and run that cell. You don't need to run them in order, and you don't need to run all of them.
4. Each chart uses Plotly, matches what's listed in your project proposal (histograms, boxplots, scatterplots, heatmaps, grouped bar charts), and is interactive — hover over points/bars to see details.

If a chart looks interesting, that's a hint about what your final project visualization section could include. You can always change the column being plotted (e.g. swap `goals` for `assists` or `xg`) — each section tells you which line to edit.


## Setup (run this first)

Loads all three seasons (2017-18, 2018-19, 2019-20), stacks them into one table, and does a couple of small cleanups:
- Standardizes the index column name across files (one file calls it `Column1`, another `Unnamed: 0`)
- Converts `value` from raw dollars/euros to **millions** (`value_m`) so it's easier to read on chart axes
- Adds a clean `season` label

Nothing here changes your actual analysis — it's just getting the data ready to plot.


In [23]:
!pip install plotly
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [24]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.options.mode.chained_assignment = None  # quiet pandas warning, optional

files = {
    "2017-18": "transfermarkt_fbref_201718.csv",
    "2018-19": "transfermarkt_fbref_201819.csv",
    "2019-20": "transfermarkt_fbref_201920.csv",
}

frames = []
for season_label, fname in files.items():
    temp = pd.read_csv(fname, sep=";", low_memory=False)
    # the first (unnamed) index column has a different name in each file - drop it safely
    first_col = temp.columns[0]
    if first_col in ("Column1", "Unnamed: 0"):
        temp = temp.drop(columns=[first_col])
    temp["season"] = season_label
    frames.append(temp)

df = pd.concat(frames, ignore_index=True)

# value is in raw currency units - convert to millions for readable axes
df["value_m"] = df["value"] / 1_000_000

# position has multiple combos like "FW,MF" - a simplified single-position version
# is handy for grouping/coloring charts
df["position_main"] = df["position"].str.split(",").str[0]

print(f"Combined shape: {df.shape[0]:,} player-seasons, {df.shape[1]} columns")
print(f"Leagues: {df['league'].unique().tolist()}")
print(f"Seasons: {df['season'].unique().tolist()}")






Combined shape: 7,108 player-seasons, 402 columns
Leagues: ['La Liga', 'Bundesliga', 'Serie A', 'Premier League', 'Ligue 1', nan]
Seasons: ['2017-18', '2018-19', '2019-20']


In [30]:
columns_to_keep = [
    'player',
    'squad',
    'league',
    'position',
    'age',
    'goals',
    'assists',
    'xg',
    'touches',
    'value_m'
]

df = df[columns_to_keep]
df.columns
df.shape

(7108, 10)

## The Menu

| # | Chart type | Good for | Jump to |
|---|---|---|---|
| 1 | Histogram | Seeing if a number (value, goals, age...) is balanced or skewed | [Section 1](#section-1) |
| 2 | Boxplot | Comparing a number across groups (leagues, positions) + spotting outliers | [Section 2](#section-2) |
| 3 | Scatterplot | Checking if two numbers move together (does more `xg` mean higher value?) | [Section 3](#section-3) |
| 4 | Correlation heatmap | Seeing which numeric stats are most related to value, all at once | [Section 4](#section-4) |
| 5 | Grouped bar chart | Comparing an average across categories (avg value per league/position) | [Section 5](#section-5) |
| 6 | Violin plot | Like a boxplot but shows the full shape of the distribution, not just quartiles | [Section 6](#section-6) |
| 7 | Bubble chart | Scatterplot + a 3rd variable shown as bubble size (e.g. age vs value, sized by goals) | [Section 7](#section-7) |
| 8 | Scatter matrix | Several scatterplots at once, to eyeball many relationships in one view | [Section 8](#section-8) |
| 9 | Line chart over seasons | Tracking how a stat changes across the three seasons for top players/teams | [Section 9](#section-9) |
| 10 | Radar chart | Comparing a single player's profile across several stats at once | [Section 10](#section-10) |

Every section is independent — run Setup, then run any one section.


<a id="section-1"></a>
## 1. Histogram — "Is this number balanced or skewed?"

**What it shows:** how often values fall into different ranges. A tall bar means lots of players share that range.

**Why it matters for this project:** your proposal flags that transfer value is probably skewed by a small number of very expensive players — a histogram is exactly how you'd confirm that visually before deciding whether to transform the variable.

**To change what's plotted:** edit the `column_to_plot` line below. Try `"age"`, `"goals"`, `"assists"`, or `"xg"`.


In [29]:
column_to_plot = "value_m"  # try: "age", "goals", "assists", "xg"

fig = px.histogram(
    df,
    x=column_to_plot,
    nbins=40,
    title=f"Distribution of {column_to_plot}",
    labels={column_to_plot: column_to_plot},
)
fig.update_layout(yaxis_title="Number of players")
fig.show()


<a id="section-2"></a>
## 2. Boxplot — "How does this number compare across groups?"

**What it shows:** for each group, the middle 50% of values (the box), the median (line in the box), and outliers (dots beyond the whiskers).

**Why it matters for this project:** your proposal specifically plans to compare transfer value across leagues, positions, and age groups — this is that chart.

**To change what's plotted:** edit `group_column` (try `"position_main"` or build an age-group column) and `value_column`.


In [27]:
group_column = "league"     # try: "position_main"
value_column = "value_m"    # try: "goals", "xg"

fig = px.box(
    df,
    x=group_column,
    y=value_column,
    title=f"{value_column} by {group_column}",
    points="outliers",  # shows outlier dots; use "all" to show every point
)
fig.show()


<a id="section-3"></a>
## 3. Scatterplot — "Do these two numbers move together?"

**What it shows:** one dot per player, positioned by two numeric variables. A clear upward or downward pattern suggests a relationship — useful for checking if a relationship looks roughly **linear**, which matters since your project plans to use linear regression.

**To change what's plotted:** edit `x_column` and `y_column`. Try `"goals"`, `"xg"`, `"touches"`, `"age"` vs `"value_m"`.


In [28]:
x_column = "xg"
y_column = "value_m"

fig = px.scatter(
    df,
    x=x_column,
    y=y_column,
    color="league",
    hover_data=["player", "squad", "season"],
    title=f"{y_column} vs {x_column}",
    opacity=0.6,
)
fig.show()


ValueError: Value of 'hover_data_2' is not the name of a column in 'data_frame'. Expected one of ['player', 'squad', 'league', 'position', 'age', 'goals', 'assists', 'xg', 'touches', 'value_m'] but received: season

<a id="section-4"></a>
## 4. Correlation Heatmap — "Which stats relate most to value, all at once?"

**What it shows:** a grid where each cell's color shows how strongly two variables move together (1 = perfectly together, -1 = perfectly opposite, 0 = unrelated). Reading the `value_m` row/column tells you, at a glance, which predictors are worth including in a regression model.

**To change what's plotted:** edit the `columns_of_interest` list — add or remove stats you want to compare.


In [ ]:
columns_of_interest = [
    "value_m", "age", "goals", "assists", "xg", "xa",
    "passes_pct", "touches", "tackles", "minutes",
]

corr_matrix = df[columns_of_interest].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation heatmap of selected stats",
)
fig.show()


<a id="section-5"></a>
## 5. Grouped Bar Chart — "What's the average per category?"

**What it shows:** one bar per category, height = the average of a chosen number. Good for a simple, very readable comparison (e.g. average player value per league).

**To change what's plotted:** edit `group_column` and `value_column`.


In [ ]:
group_column = "league"   # try: "position_main"
value_column = "value_m"

avg_by_group = (
    df.groupby(group_column)[value_column]
    .mean()
    .reset_index()
    .sort_values(value_column, ascending=False)
)

fig = px.bar(
    avg_by_group,
    x=group_column,
    y=value_column,
    title=f"Average {value_column} by {group_column}",
    text_auto=".1f",
)
fig.show()


<a id="section-6"></a>
## 6. Violin Plot — "Boxplot, but show me the actual shape"

**What it shows:** like a boxplot, but instead of just quartiles, it draws the full distribution shape (wider = more players at that value). Good when you suspect a group has two clusters (e.g. a few stars and many average-value players) that a boxplot would hide.

**To change what's plotted:** edit `group_column` and `value_column`.


In [ ]:
group_column = "league"
value_column = "value_m"

fig = px.violin(
    df,
    x=group_column,
    y=value_column,
    box=True,        # shows a mini boxplot inside the violin
    points=False,
    title=f"Distribution shape of {value_column} by {group_column}",
)
fig.show()


<a id="section-7"></a>
## 7. Bubble Chart — "Scatterplot + a 3rd number shown as size"

**What it shows:** same as a scatterplot, but bubble size encodes a third variable, so you can compare three numeric relationships at once without a 3D plot.

**To change what's plotted:** edit `x_column`, `y_column`, and `size_column`.


In [ ]:
x_column = "age"
y_column = "value_m"
size_column = "goals"

# bubble size can't be negative - filter those out, and sample for readability
plot_df = df[df[size_column] >= 0].sample(n=min(500, len(df)), random_state=1)

fig = px.scatter(
    plot_df,
    x=x_column,
    y=y_column,
    size=size_column,
    color="league",
    hover_data=["player", "squad"],
    title=f"{y_column} vs {x_column}, bubble size = {size_column}",
    opacity=0.6,
    size_max=30,
)
fig.show()


<a id="section-8"></a>
## 8. Scatter Matrix — "Show me several relationships at once"

**What it shows:** a grid of mini scatterplots, every variable plotted against every other variable in your list. Great for a quick first scan before picking which single relationships to dig into with Section 3.

**To change what's plotted:** edit `columns_of_interest` — keep it to 4-5 columns or the grid gets crowded.


In [ ]:
columns_of_interest = ["value_m", "age", "goals", "xg"]

# sampling keeps the plot from getting too dense / slow to render
plot_df = df.sample(n=min(400, len(df)), random_state=1)

fig = px.scatter_matrix(
    plot_df,
    dimensions=columns_of_interest,
    color="league",
    title="Scatter matrix of selected stats",
    opacity=0.6,
)
fig.update_traces(diagonal_visible=False)
fig.show()


<a id="section-9"></a>
## 9. Line Chart Over Seasons — "How did this change over time?"

**What it shows:** how a stat trends across the three seasons in this dataset, for a handful of players. Useful if your project wants to discuss value changes over time rather than just a single-season snapshot.

**To change what's plotted:** edit `players_to_track` (must match the `player` column spelling) and `value_column`.


In [ ]:
players_to_track = ["Lionel Messi", "Cristiano Ronaldo", "Kylian Mbappé"]
value_column = "value_m"

plot_df = df[df["player"].isin(players_to_track)].sort_values("season")

fig = px.line(
    plot_df,
    x="season",
    y=value_column,
    color="player",
    markers=True,
    title=f"{value_column} across seasons",
)
fig.show()

# Tip: not sure which names exist in the data? Run this to search:
# df[df['player'].str.contains('Messi', case=False, na=False)]['player'].unique()


<a id="section-10"></a>
## 10. Radar Chart — "Show one player's profile across several stats"

**What it shows:** a single player's stats plotted on several axes radiating from a center point, scaled 0-100 (percentile vs. everyone else in the dataset) so very different stats (goals vs. pass completion %) can share one chart.

**To change what's plotted:** edit `player_name` and/or `stats_to_compare`.


In [ ]:
player_name = "Lionel Messi"
stats_to_compare = ["goals", "assists", "xg", "xa", "passes_pct", "dribbles_completed"]

# convert each stat to a percentile (0-100) so different scales are comparable
percentiles = df[stats_to_compare].rank(pct=True) * 100
percentiles["player"] = df["player"].values

player_row = percentiles[percentiles["player"] == player_name]

if player_row.empty:
    print(f"'{player_name}' not found. Try searching: df[df['player'].str.contains('messi', case=False, na=False)]['player'].unique()")
else:
    values = player_row[stats_to_compare].iloc[0].tolist()
    values.append(values[0])  # close the loop
    labels = stats_to_compare + [stats_to_compare[0]]

    fig = go.Figure(data=go.Scatterpolar(r=values, theta=labels, fill="toself", name=player_name))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
        title=f"{player_name} — percentile rank across selected stats",
    )
    fig.show()


## Picking a final set

For a class project, you generally don't need all ten — a strong mix is usually:
- **1 histogram** (check the shape of your outcome variable, `value_m`)
- **1 boxplot or violin** (compare value across a category like league or position)
- **1-2 scatterplots** (show relationships with your strongest predictors)
- **1 correlation heatmap** (justify which predictors you chose for the regression)

The grouped bar chart, bubble chart, scatter matrix, line chart, and radar chart are good *extras* if you want one more visual that tells a specific story (e.g. "here's how a star player's value changed over time").
